In [ ]:
'''
    Warren-Cowley short-range order (WC-SRO) calculation for NCM layered oxide.

    Created on Jul 26, 2024 at RISM (Shinshu University)
    Last update: Jul 30, 2026 17:22 JST

    Copyright © 2022-2026 Quang Nguyen. All rights reserved.
'''

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, BoundaryNorm
from ase import Atoms
from ase.io import read
from ase.neighborlist import NeighborList
from collections import defaultdict

# Function to calculate WC-SRO
def warren_cowley_sro(atoms, elements, cutoff_min, cutoff_max, attribute):
    symbols = atoms.get_chemical_symbols()
    n_atoms = len(symbols)
    element_counts = defaultdict(int)
    for symbol in symbols:
        if symbol in elements:
            element_counts[symbol] += 1
    n_atoms_x = sum(element_counts.values())
    concentrations = {el: count / n_atoms_x for el, count in element_counts.items()}
    cutoffs_min = [cutoff_min / 2] * n_atoms
    nl_min = NeighborList(cutoffs_min, skin=0.0, self_interaction=False, bothways=True)
    nl_min.update(atoms)
    cutoffs_max = [cutoff_max / 2] * n_atoms
    nl_max = NeighborList(cutoffs_max, skin=0.0, self_interaction=False, bothways=True)
    nl_max.update(atoms)
    total_neighbors = defaultdict(int)
    element_pair_neighbors = defaultdict(lambda: defaultdict(int))
    for i in range(n_atoms):
        if symbols[i] in elements:
            indices_min, _ = nl_min.get_neighbors(i)
            indices_max, _ = nl_max.get_neighbors(i)
            valid_neighbors = set(indices_max) - set(indices_min)
            for j in valid_neighbors:
                if symbols[j] in elements:
                    total_neighbors[symbols[i]] += 1
                    element_pair_neighbors[symbols[i]][symbols[j]] += 1
    sro_parameters = defaultdict(dict)
    for el_i in elements:
        for el_j in elements:
            if total_neighbors[el_i] > 0:
                if attribute == 'global':
                    P_ij = element_pair_neighbors[el_i][el_j] / sum(total_neighbors.values())
                elif attribute == 'local':
                    P_ij = element_pair_neighbors[el_i][el_j] / total_neighbors[el_i]
                else:
                    raise ValueError("Unsupported attribute! Use 'global' or 'local'.")
            else:
                P_ij = 0
            if concentrations[el_j] > 0:
                sro_parameters[el_i][el_j] = 1 - (P_ij / concentrations[el_j])
            else:
                sro_parameters[el_i][el_j] = 0
    return sro_parameters

# Function to create WC-SRO matrix
def create_sro_matrix(sro_parameters, elements):
    sro_matrix = np.zeros((len(elements), len(elements)))
    for i, el_i in enumerate(elements):
        for j, el_j in enumerate(elements):
            sro_matrix[i, j] = sro_parameters[el_i][el_j]

    return sro_matrix

# =================================================================================================

# User settings
compound    = 'NCM523'
temperature = '50K'
elements    = ['Ni', 'Co', 'Mn']
struct_file = '../Other_Structures/'+compound+'_5x4x1_'+temperature+'.cif'
atoms       = read(struct_file)
a, b, c     = atoms.cell.lengths()[0], atoms.cell.lengths()[1], atoms.cell.lengths()[2]
alpha, beta, gamma = atoms.cell.angles()[0], atoms.cell.angles()[1], atoms.cell.angles()[2]
c_new       = c + 1.0
atoms.set_cell([a, b, c_new, alpha, beta, gamma], scale_atoms=True)

# Calculate Warren Cowley SRO for NCM
cutoff_1NN = 3.000
cutoff_2NN = 5.100
cutoff_3NN = 5.400
sro_parameters_1NN = warren_cowley_sro(atoms, elements, 0, cutoff_1NN, 'local')
sro_matrix_1NN = create_sro_matrix(sro_parameters_1NN, elements)
sro_parameters_2NN = warren_cowley_sro(atoms, elements, cutoff_1NN, cutoff_2NN, 'local')
sro_matrix_2NN = create_sro_matrix(sro_parameters_2NN, elements)
sro_parameters_3NN = warren_cowley_sro(atoms, elements, cutoff_2NN, cutoff_3NN, 'local')
sro_matrix_3NN = create_sro_matrix(sro_parameters_3NN, elements)
sro_df_1NN = pd.DataFrame(sro_matrix_1NN, index=elements, columns=elements)
sro_df_1NN = sro_df_1NN.reindex(index=list(reversed(elements)), columns=elements)
sro_df_2NN = pd.DataFrame(sro_matrix_2NN, index=elements, columns=elements)
sro_df_2NN = sro_df_2NN.reindex(index=list(reversed(elements)), columns=elements)
sro_df_3NN = pd.DataFrame(sro_matrix_3NN, index=elements, columns=elements)
sro_df_3NN = sro_df_3NN.reindex(index=list(reversed(elements)), columns=elements)
all_data = np.concatenate([sro_matrix_1NN.flatten(), sro_matrix_2NN.flatten(), sro_matrix_3NN.flatten()])
vmin, vmax = -2.0, 1.0 

# Plot 1NN and 2NN in one plot
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=False)
sns.set(font_scale=2.0)
sns.heatmap(sro_df_1NN, annot=True, fmt='.2f', cmap='coolwarm', vmin=vmin, vmax=vmax, cbar=False, linewidths=0.5, ax=axes[0])
sns.heatmap(sro_df_2NN, annot=True, fmt='.2f', cmap='coolwarm', vmin=vmin, vmax=vmax, cbar=False, linewidths=0.5, ax=axes[1])
axes[0].set_title('WC-SRO : 1NN')
axes[1].set_title('WC-SRO : 2NN')
axes[0].tick_params(left=True, bottom=True, labelbottom=True, labeltop=False, labelsize=22)
axes[1].tick_params(left=True, bottom=True, labelbottom=True, labeltop=False, labelsize=22)
for ax in axes:
    ax.set_aspect('equal')
fig.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
plt.show()